In [ ]:
# Importe
from pathlib import Path
import json
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    import pm4py
except ImportError as e:
    raise ImportError('PM4Py ist nicht installiert. Bitte in der .venv ausführen: pip install -U pm4py') from e
print('Imports OK')
print('Python:', platform.python_version())
print('pandas:', pd.__version__)


In [ ]:
# Pfade und Einstellungen
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'descriptive_process_analysis'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
PRIMARY_LABEL = 'label_scd_p90_or'
ROBUSTNESS_LABEL = 'label_scd_p95_or'
SECONDARY_LABELS = ['label_temporal_duration_p90', 'label_path_change_or_objection', 'label_remove_document', 'label_scd_p90_and']
BUILD_ACTIVITY_VARIANTS = True
BUILD_COMBINED_VARIANTS = False
BUILD_DIRECT_FOLLOWS = True
BUILD_COMBINED_DIRECT_FOLLOWS = False
print('Project root:', PROJECT_ROOT)
print('Expected log path:', LOG_PATH)
print('Log exists:', LOG_PATH.exists())
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Datensatzpfad suchen
if not LOG_PATH.exists():
    candidates = sorted(DATA_RAW.glob('*.xes*')) + sorted(DATA_RAW.glob('**/*.xes*'))
    print('Gefundene XES-Kandidaten:')
    for c in candidates[:20]:
        print('-', c)
    if candidates:
        LOG_PATH = candidates[0]
        print('Nutze automatisch:', LOG_PATH)
    else:
        raise FileNotFoundError(f'Keine XES/XES.GZ-Datei in {DATA_RAW} gefunden.')


In [ ]:
# Hilfsfunktionen
created_tables = []
created_figures = []

def save_csv(obj, filename, index=True):
    path = TABLE_DIR / filename
    if isinstance(obj, pd.Series):
        obj.to_frame().to_csv(path, index=index, encoding='utf-8-sig')
    else:
        obj.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def pct(part, whole):
    if whole == 0 or pd.isna(whole):
        return np.nan
    return float(part) / float(whole) * 100

def safe_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def standardised_mean_diff(x_pos, x_neg):
    x_pos = pd.to_numeric(pd.Series(x_pos), errors='coerce').dropna()
    x_neg = pd.to_numeric(pd.Series(x_neg), errors='coerce').dropna()
    if len(x_pos) < 2 or len(x_neg) < 2:
        return np.nan
    pooled = np.sqrt((x_pos.var(ddof=1) + x_neg.var(ddof=1)) / 2)
    if pooled == 0 or pd.isna(pooled):
        return np.nan
    return (x_pos.mean() - x_neg.mean()) / pooled

def describe_bool_share(s):
    return round(float(pd.Series(s).mean() * 100), 4)
print('Helper OK')


In [ ]:
# Log laden
print('Lade Log. Das kann beim BPIC-2018-Log einige Zeit dauern ...')
raw_log = pm4py.read_xes(str(LOG_PATH))
if isinstance(raw_log, pd.DataFrame):
    df = raw_log.copy()
else:
    df = pm4py.convert_to_dataframe(raw_log)
print('DataFrame shape:', df.shape)
df.head()


In [ ]:
# Kernspalten vorbereiten
CASE_COL = safe_first_existing(df.columns, ['case:concept:name', 'case_id', 'case'])
ACTIVITY_COL = safe_first_existing(df.columns, ['concept:name', 'activity'])
TIME_COL = safe_first_existing(df.columns, ['time:timestamp', 'timestamp', 'time'])
RAW_ACTIVITY_COL = 'activity' if 'activity' in df.columns else ACTIVITY_COL
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise RuntimeError(f'Kernspalten fehlen. CASE_COL={CASE_COL}, ACTIVITY_COL={ACTIVITY_COL}, TIME_COL={TIME_COL}')
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True)
if '__row_order__' not in df.columns:
    df['__row_order__'] = np.arange(len(df), dtype=np.int64)
for col in ['doctype', 'subprocess', RAW_ACTIVITY_COL]:
    if col not in df.columns:
        df[col] = 'unknown'
if 'combined_activity' not in df.columns:
    df['combined_activity'] = df['doctype'].astype(str) + ' | ' + df['subprocess'].astype(str) + ' | ' + df[RAW_ACTIVITY_COL].astype(str)
CASE_YEAR_COL = safe_first_existing(df.columns, ['case:year', 'year', 'case:subsidy_year', 'case:application_year'])
CASE_DEPARTMENT_COL = safe_first_existing(df.columns, ['case:department', 'department'])
CASE_APPLICANT_COL = safe_first_existing(df.columns, ['case:applicant', 'applicant'])
CASE_REJECTED_COL = safe_first_existing(df.columns, ['case:rejected', 'rejected'])
basic_info = {'log_path': str(LOG_PATH), 'events': int(len(df)), 'cases': int(df[CASE_COL].nunique()), 'columns': int(df.shape[1]), 'activities': int(df[ACTIVITY_COL].nunique()), 'combined_activities': int(df['combined_activity'].nunique()), 'timestamp_min': str(df[TIME_COL].min()), 'timestamp_max': str(df[TIME_COL].max()), 'case_col': CASE_COL, 'activity_col': ACTIVITY_COL, 'raw_activity_col': RAW_ACTIVITY_COL, 'time_col': TIME_COL, 'case_year_col': CASE_YEAR_COL, 'case_department_col': CASE_DEPARTMENT_COL, 'case_applicant_col': CASE_APPLICANT_COL, 'case_rejected_col': CASE_REJECTED_COL, 'memory_mb': round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2)}
save_json(basic_info, '00_basic_info.json')
print(json.dumps(basic_info, indent=2, ensure_ascii=False))


In [ ]:
# Spaltenprofil
column_profile = pd.DataFrame({'column': df.columns, 'dtype': [str(df[c].dtype) for c in df.columns], 'missing_count': [int(df[c].isna().sum()) for c in df.columns], 'missing_pct': [round(float(df[c].isna().mean() * 100), 4) for c in df.columns], 'nunique': [int(df[c].nunique(dropna=False)) for c in df.columns]})
save_csv(column_profile, '01_column_profile.csv', index=False)
column_profile.head(20)


In [ ]:
# Fallmerkmale
print('Baue Case-Level-Tabelle ...')
g = df.groupby(CASE_COL, sort=False)
case_df = g[TIME_COL].agg(case_start='min', case_end='max')
case_df['duration_days'] = (case_df['case_end'] - case_df['case_start']).dt.total_seconds() / (3600 * 24)
case_df['event_count'] = g.size().astype(int)
case_attr_cols = [c for c in df.columns if c.startswith('case:') and c != CASE_COL]
if case_attr_cols:
    case_attrs_df = g[case_attr_cols].first()
    case_df = case_df.join(case_attrs_df)
for col, out_col in [(ACTIVITY_COL, 'n_activity_labels'), (RAW_ACTIVITY_COL, 'n_raw_activity_labels'), ('combined_activity', 'n_combined_activity_labels'), ('doctype', 'n_doctypes'), ('subprocess', 'n_subprocesses'), ('docid', 'n_docids'), ('docid_uuid', 'n_docid_uuids'), ('org:resource', 'n_resources')]:
    if col in df.columns:
        case_df[out_col] = g[col].nunique(dropna=True).astype(int)
case_df['case_start_calendar_year'] = case_df['case_start'].dt.year
case_df['case_end_calendar_year'] = case_df['case_end'].dt.year
print('Case-Level shape:', case_df.shape)
case_df.head()


In [ ]:
# Wiederholungsmerkmale
print('Berechne Rework-Metriken ...')
raw_counts = df.groupby([CASE_COL, ACTIVITY_COL], sort=False).size().rename('count')
raw_rework_extra = raw_counts.sub(1).clip(lower=0).groupby(level=0).sum()
case_df['raw_rework_extra'] = raw_rework_extra.reindex(case_df.index).fillna(0).astype(int)
combined_counts = df.groupby([CASE_COL, 'combined_activity'], sort=False).size().rename('count')
combined_rework_extra = combined_counts.sub(1).clip(lower=0).groupby(level=0).sum()
case_df['combined_rework_extra'] = combined_rework_extra.reindex(case_df.index).fillna(0).astype(int)
case_df['max_repeat_raw_activity'] = raw_counts.groupby(level=0).max().reindex(case_df.index).fillna(0).astype(int)
case_df['max_repeat_combined_activity'] = combined_counts.groupby(level=0).max().reindex(case_df.index).fillna(0).astype(int)
case_df['combined_rework_share_of_events'] = case_df['combined_rework_extra'] / case_df['event_count'].replace(0, np.nan)
case_df['raw_rework_share_of_events'] = case_df['raw_rework_extra'] / case_df['event_count'].replace(0, np.nan)
case_df[['event_count', 'raw_rework_extra', 'combined_rework_extra', 'combined_rework_share_of_events']].describe()


In [ ]:
# Pfadmerkmale
print('Berechne Pfad-/Exception-Flags ...')
subprocess_s = df['subprocess'].astype(str).str.lower()
activity_s = df[RAW_ACTIVITY_COL].astype(str).str.lower()
concept_s = df[ACTIVITY_COL].astype(str).str.lower()
combined_s = df['combined_activity'].astype(str).str.lower()
note_s = df['note'].astype(str).str.lower() if 'note' in df.columns else pd.Series('', index=df.index)

def add_case_flag(flag_name, event_mask):
    cases = df.loc[event_mask.fillna(False), CASE_COL].dropna().unique()
    case_df[flag_name] = case_df.index.isin(cases)
    return int(case_df[flag_name].sum())
add_case_flag('flag_path_change', subprocess_s.eq('change'))
add_case_flag('flag_path_objection', subprocess_s.eq('objection'))
case_df['flag_path_change_or_objection'] = case_df['flag_path_change'] | case_df['flag_path_objection']
add_case_flag('flag_remove_document', activity_s.str.contains('remove', regex=False) & activity_s.str.contains('document', regex=False))
add_case_flag('flag_payment_abort', activity_s.str.contains('abort', regex=False) & activity_s.str.contains('payment', regex=False))
add_case_flag('flag_revoke_decision', activity_s.str.contains('revoke', regex=False) & activity_s.str.contains('decision', regex=False))
add_case_flag('flag_refuse', activity_s.str.contains('refuse', regex=False) | concept_s.str.contains('refuse', regex=False))
add_case_flag('flag_withdraw', activity_s.str.contains('withdraw', regex=False) | concept_s.str.contains('withdraw', regex=False))
add_case_flag('flag_restart_editing', activity_s.str.contains('begin editing', regex=False) & note_s.str.contains('restart', regex=False))
if CASE_REJECTED_COL is not None and CASE_REJECTED_COL in case_df.columns:
    case_df['flag_case_rejected'] = case_df[CASE_REJECTED_COL].astype(str).str.lower().isin(['true', '1', 'yes'])
else:
    case_df['flag_case_rejected'] = False
flag_cols = [c for c in case_df.columns if c.startswith('flag_')]
flag_summary = pd.DataFrame({'flag': flag_cols, 'cases_true': [int(case_df[c].sum()) for c in flag_cols], 'share_pct': [round(float(case_df[c].mean() * 100), 4) for c in flag_cols]}).sort_values('share_pct', ascending=False)
save_csv(flag_summary, '02_path_exception_flag_summary.csv', index=False)
flag_summary


In [ ]:
# Labels
print('Berechne Labels ...')
thresholds = {'duration_p90': float(case_df['duration_days'].quantile(0.9)), 'duration_p95': float(case_df['duration_days'].quantile(0.95)), 'event_count_p90': float(case_df['event_count'].quantile(0.9)), 'event_count_p95': float(case_df['event_count'].quantile(0.95)), 'combined_rework_p90': float(case_df['combined_rework_extra'].quantile(0.9)), 'combined_rework_p95': float(case_df['combined_rework_extra'].quantile(0.95))}
case_df['label_temporal_duration_p90'] = case_df['duration_days'] >= thresholds['duration_p90']
case_df['label_temporal_duration_p95'] = case_df['duration_days'] >= thresholds['duration_p95']
case_df['label_structural_many_events_p90'] = case_df['event_count'] >= thresholds['event_count_p90']
case_df['label_structural_many_events_p95'] = case_df['event_count'] >= thresholds['event_count_p95']
case_df['label_structural_high_combined_rework_p90'] = case_df['combined_rework_extra'] >= thresholds['combined_rework_p90']
case_df['label_structural_high_combined_rework_p95'] = case_df['combined_rework_extra'] >= thresholds['combined_rework_p95']
case_df['label_scd_p90_or'] = case_df['label_structural_many_events_p90'] | case_df['label_structural_high_combined_rework_p90']
case_df['label_scd_p95_or'] = case_df['label_structural_many_events_p95'] | case_df['label_structural_high_combined_rework_p95']
case_df['label_scd_p90_and'] = case_df['label_structural_many_events_p90'] & case_df['label_structural_high_combined_rework_p90']
case_df['label_scd_p95_and'] = case_df['label_structural_many_events_p95'] & case_df['label_structural_high_combined_rework_p95']
case_df['label_path_change_or_objection'] = case_df['flag_path_change_or_objection']
case_df['label_path_change'] = case_df['flag_path_change']
case_df['label_path_objection'] = case_df['flag_path_objection']
case_df['label_remove_document'] = case_df['flag_remove_document']
case_df['label_payment_abort'] = case_df['flag_payment_abort']
case_df['label_revoke_decision'] = case_df['flag_revoke_decision']
case_df['label_case_rejected'] = case_df['flag_case_rejected']
label_cols = [c for c in case_df.columns if c.startswith('label_')]
label_prevalence = pd.DataFrame({'label': label_cols, 'cases_true': [int(case_df[c].sum()) for c in label_cols], 'share_pct': [round(float(case_df[c].mean() * 100), 4) for c in label_cols]}).sort_values('share_pct', ascending=False)
save_json(thresholds, '03_label_thresholds.json')
save_csv(label_prevalence, '04_label_prevalence.csv', index=False)
print('Thresholds:')
print(json.dumps(thresholds, indent=2))
label_prevalence


In [ ]:
# Plausibilitätsprüfung
expected_notes = []
expected = {'events': 2514266, 'cases': 43809, 'activities': 41}
actual = {'events': len(df), 'cases': df[CASE_COL].nunique(), 'activities': df[ACTIVITY_COL].nunique()}
for key in expected:
    ok = expected[key] == actual[key]
    expected_notes.append({'metric': key, 'expected': expected[key], 'actual': int(actual[key]), 'matches_expected': ok})
expected_notes.append({'metric': 'primary_label_cases', 'expected': 'ca. 5373', 'actual': int(case_df[PRIMARY_LABEL].sum()), 'matches_expected': abs(int(case_df[PRIMARY_LABEL].sum()) - 5373) <= 20})
expected_notes.append({'metric': 'primary_label_share_pct', 'expected': 'ca. 12.26', 'actual': round(float(case_df[PRIMARY_LABEL].mean() * 100), 4), 'matches_expected': abs(float(case_df[PRIMARY_LABEL].mean() * 100) - 12.26) <= 0.2})
validation_df = pd.DataFrame(expected_notes)
save_csv(validation_df, '05_validation_against_previous_audit.csv', index=False)
validation_df


In [ ]:
# Gesamtübersicht
case_summary_metrics = ['duration_days', 'event_count', 'raw_rework_extra', 'combined_rework_extra', 'combined_rework_share_of_events', 'n_activity_labels', 'n_combined_activity_labels', 'n_doctypes', 'n_subprocesses', 'n_docids', 'n_resources']
case_summary_metrics = [c for c in case_summary_metrics if c in case_df.columns]
case_level_summary = case_df[case_summary_metrics].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T
save_csv(case_level_summary, '06_case_level_summary.csv')
case_level_summary


In [ ]:
# Verteilung nach Jahr und Bereich
summary_tables = {}
if CASE_YEAR_COL is not None and CASE_YEAR_COL in case_df.columns:
    summary_tables['case_year_counts'] = case_df[CASE_YEAR_COL].value_counts(dropna=False).sort_index().rename('case_count')
    save_csv(summary_tables['case_year_counts'], '07_case_year_counts.csv')
if CASE_DEPARTMENT_COL is not None and CASE_DEPARTMENT_COL in case_df.columns:
    summary_tables['case_department_counts'] = case_df[CASE_DEPARTMENT_COL].value_counts(dropna=False).rename('case_count')
    save_csv(summary_tables['case_department_counts'], '08_case_department_counts.csv')
start_year_counts = case_df['case_start_calendar_year'].value_counts(dropna=False).sort_index().rename('case_count')
end_year_counts = case_df['case_end_calendar_year'].value_counts(dropna=False).sort_index().rename('case_count')
save_csv(start_year_counts, '09_case_start_calendar_year_counts.csv')
save_csv(end_year_counts, '10_case_end_calendar_year_counts.csv')
print('case:year counts:' if CASE_YEAR_COL else 'Keine case:year-Spalte gefunden')
if CASE_YEAR_COL:
    display(summary_tables['case_year_counts'].to_frame())
print('case start calendar year counts:')
display(start_year_counts.to_frame())


In [ ]:
# Standardfälle und SCD
compare_metrics = ['duration_days', 'event_count', 'combined_rework_extra', 'raw_rework_extra', 'combined_rework_share_of_events', 'n_doctypes', 'n_subprocesses', 'n_docids', 'n_resources', 'n_activity_labels', 'n_combined_activity_labels', 'max_repeat_combined_activity', 'max_repeat_raw_activity']
compare_metrics = [c for c in compare_metrics if c in case_df.columns]
rows = []
for m in compare_metrics:
    pos = pd.to_numeric(case_df.loc[case_df[PRIMARY_LABEL], m], errors='coerce')
    neg = pd.to_numeric(case_df.loc[~case_df[PRIMARY_LABEL], m], errors='coerce')
    rows.append({'metric': m, 'n_positive': int(pos.notna().sum()), 'n_negative': int(neg.notna().sum()), 'mean_positive': round(float(pos.mean()), 4), 'mean_negative': round(float(neg.mean()), 4), 'median_positive': round(float(pos.median()), 4), 'median_negative': round(float(neg.median()), 4), 'p90_positive': round(float(pos.quantile(0.9)), 4), 'p90_negative': round(float(neg.quantile(0.9)), 4), 'median_difference': round(float(pos.median() - neg.median()), 4), 'median_ratio_pos_neg': round(float(pos.median() / neg.median()), 4) if neg.median() not in [0, np.nan] and neg.median() != 0 else np.nan, 'smd_mean_pos_neg': round(float(standardised_mean_diff(pos, neg)), 4) if not pd.isna(standardised_mean_diff(pos, neg)) else np.nan})
group_comparison = pd.DataFrame(rows).sort_values('smd_mean_pos_neg', key=lambda s: s.abs(), ascending=False)
save_csv(group_comparison, '11_group_comparison_primary_label.csv', index=False)
group_comparison


In [ ]:
# Sekundäre Pfadlabels
secondary_flags_for_compare = ['label_temporal_duration_p90', 'label_temporal_duration_p95', 'label_path_change_or_objection', 'label_path_change', 'label_path_objection', 'label_remove_document', 'label_payment_abort', 'label_revoke_decision', 'label_case_rejected', 'flag_refuse', 'flag_withdraw', 'flag_restart_editing']
secondary_flags_for_compare = [c for c in secondary_flags_for_compare if c in case_df.columns and c != PRIMARY_LABEL]
rows = []
for c in secondary_flags_for_compare:
    pos_share = case_df.loc[case_df[PRIMARY_LABEL], c].mean() * 100
    neg_share = case_df.loc[~case_df[PRIMARY_LABEL], c].mean() * 100
    rows.append({'indicator': c, 'share_in_scd_pct': round(float(pos_share), 4), 'share_in_standard_pct': round(float(neg_share), 4), 'difference_pp': round(float(pos_share - neg_share), 4), 'ratio_scd_to_standard': round(float(pos_share / neg_share), 4) if neg_share != 0 else np.nan})
secondary_overlap_primary = pd.DataFrame(rows).sort_values('difference_pp', ascending=False)
save_csv(secondary_overlap_primary, '12_secondary_indicators_by_primary_label.csv', index=False)
secondary_overlap_primary


In [ ]:
# Labelanteile nach Jahr und Bereich
def label_share_by_category(data, category_col, label_col, filename):
    if category_col is None or category_col not in data.columns:
        return pd.DataFrame()
    out = data.groupby(category_col).agg(n_cases=(label_col, 'size'), n_positive=(label_col, 'sum'), share_positive_pct=(label_col, lambda x: round(float(x.mean() * 100), 4)), median_duration_days=('duration_days', 'median'), median_event_count=('event_count', 'median'), median_combined_rework_extra=('combined_rework_extra', 'median')).reset_index()
    save_csv(out, filename, index=False)
    return out
label_by_case_year = label_share_by_category(case_df, CASE_YEAR_COL, PRIMARY_LABEL, '13_primary_label_by_case_year.csv')
label_by_department = label_share_by_category(case_df, CASE_DEPARTMENT_COL, PRIMARY_LABEL, '14_primary_label_by_department.csv')
label_by_start_year = label_share_by_category(case_df, 'case_start_calendar_year', PRIMARY_LABEL, '15_primary_label_by_start_calendar_year.csv')
print('Primary Label by case:year')
display(label_by_case_year)
print('Primary Label by department')
display(label_by_department.head(20))


In [ ]:
# Jährliche Labelanteile
year_label_cols = [PRIMARY_LABEL, ROBUSTNESS_LABEL] + SECONDARY_LABELS
year_label_cols = list(dict.fromkeys([c for c in year_label_cols if c in case_df.columns]))
if CASE_YEAR_COL is not None and CASE_YEAR_COL in case_df.columns:
    year_label_summary = case_df.groupby(CASE_YEAR_COL).agg(n_cases=(PRIMARY_LABEL, 'size'), **{f'{c}_share_pct': (c, lambda x: round(float(x.mean() * 100), 4)) for c in year_label_cols}).reset_index()
    save_csv(year_label_summary, '16_key_labels_by_case_year.csv', index=False)
    display(year_label_summary)
else:
    year_label_summary = pd.DataFrame()
    print('Keine case:year-Spalte verfügbar.')


In [ ]:
# Ereignisdaten mit Primärlabel
label_map = case_df[PRIMARY_LABEL].astype(bool)
event_work_cols = [CASE_COL, TIME_COL, ACTIVITY_COL, RAW_ACTIVITY_COL, 'combined_activity', 'doctype', 'subprocess', '__row_order__']
for optional_col in ['org:resource', 'docid', 'docid_uuid', CASE_YEAR_COL, CASE_DEPARTMENT_COL]:
    if optional_col is not None and optional_col in df.columns and (optional_col not in event_work_cols):
        event_work_cols.append(optional_col)
event_df = df[event_work_cols].copy()
event_df[PRIMARY_LABEL] = event_df[CASE_COL].map(label_map).astype(bool)
print(event_df.shape)
event_df.head()


In [ ]:
# Aktivitätshäufigkeiten
activity_event_counts = event_df.groupby([PRIMARY_LABEL, ACTIVITY_COL]).size().reset_index(name='event_count')
activity_event_counts['group_event_total'] = activity_event_counts.groupby(PRIMARY_LABEL)['event_count'].transform('sum')
activity_event_counts['event_share_within_group_pct'] = activity_event_counts['event_count'] / activity_event_counts['group_event_total'] * 100
save_csv(activity_event_counts.sort_values([PRIMARY_LABEL, 'event_count'], ascending=[True, False]), '17_activity_event_counts_by_primary_label.csv', index=False)
combined_event_counts = event_df.groupby([PRIMARY_LABEL, 'combined_activity']).size().reset_index(name='event_count')
combined_event_counts['group_event_total'] = combined_event_counts.groupby(PRIMARY_LABEL)['event_count'].transform('sum')
combined_event_counts['event_share_within_group_pct'] = combined_event_counts['event_count'] / combined_event_counts['group_event_total'] * 100
save_csv(combined_event_counts.sort_values([PRIMARY_LABEL, 'event_count'], ascending=[True, False]).head(1000), '18_top_combined_activity_event_counts_by_primary_label.csv', index=False)
presence = event_df[[CASE_COL, PRIMARY_LABEL, 'combined_activity']].drop_duplicates()
case_totals_by_group = case_df.groupby(PRIMARY_LABEL).size().rename('n_cases_group')
combined_case_presence = presence.groupby([PRIMARY_LABEL, 'combined_activity'])[CASE_COL].nunique().reset_index(name='case_count')
combined_case_presence['n_cases_group'] = combined_case_presence[PRIMARY_LABEL].map(case_totals_by_group)
combined_case_presence['case_share_within_group_pct'] = combined_case_presence['case_count'] / combined_case_presence['n_cases_group'] * 100
wide_presence = combined_case_presence.pivot(index='combined_activity', columns=PRIMARY_LABEL, values='case_share_within_group_pct').fillna(0)
rename_cols = {}
for c in wide_presence.columns:
    if bool(c) is False:
        rename_cols[c] = 'standard_share_pct'
    else:
        rename_cols[c] = 'scd_share_pct'
wide_presence = wide_presence.rename(columns=rename_cols)
if 'standard_share_pct' not in wide_presence.columns:
    wide_presence['standard_share_pct'] = 0
if 'scd_share_pct' not in wide_presence.columns:
    wide_presence['scd_share_pct'] = 0
wide_presence['difference_pp_scd_minus_standard'] = wide_presence['scd_share_pct'] - wide_presence['standard_share_pct']
wide_presence['ratio_scd_to_standard'] = wide_presence['scd_share_pct'] / wide_presence['standard_share_pct'].replace(0, np.nan)
wide_presence = wide_presence.reset_index().sort_values('difference_pp_scd_minus_standard', ascending=False)
save_csv(wide_presence, '19_combined_activity_case_presence_difference_primary_label.csv', index=False)
wide_presence.head(30)


In [ ]:
# Dokumenttypen und Teilprozesse
for col, filename in [('doctype', '20_doctype_event_counts_by_primary_label.csv'), ('subprocess', '21_subprocess_event_counts_by_primary_label.csv')]:
    if col in event_df.columns:
        tmp = event_df.groupby([PRIMARY_LABEL, col]).size().reset_index(name='event_count')
        tmp['group_event_total'] = tmp.groupby(PRIMARY_LABEL)['event_count'].transform('sum')
        tmp['event_share_within_group_pct'] = tmp['event_count'] / tmp['group_event_total'] * 100
        save_csv(tmp.sort_values([PRIMARY_LABEL, 'event_count'], ascending=[True, False]), filename, index=False)
        display(tmp.sort_values([PRIMARY_LABEL, 'event_count'], ascending=[True, False]).head(30))
if 'doctype' in event_df.columns and 'subprocess' in event_df.columns:
    matrix_all = pd.crosstab(event_df['doctype'], event_df['subprocess'])
    save_csv(matrix_all, '22_matrix_doctype_subprocess_all_events.csv')
    for label_value, name in [(False, 'standard'), (True, 'scd')]:
        m = pd.crosstab(event_df.loc[event_df[PRIMARY_LABEL] == label_value, 'doctype'], event_df.loc[event_df[PRIMARY_LABEL] == label_value, 'subprocess'])
        save_csv(m, f'23_matrix_doctype_subprocess_{name}_events.csv')
    display(matrix_all)


In [ ]:
# Varianten
if BUILD_ACTIVITY_VARIANTS:
    print('Baue Activity-Varianten. Das kann dauern ...')
    seq_df = df[[CASE_COL, TIME_COL, '__row_order__', ACTIVITY_COL, 'combined_activity']].sort_values([CASE_COL, TIME_COL, '__row_order__'], kind='mergesort')
    activity_variant_series = seq_df.groupby(CASE_COL, sort=False)[ACTIVITY_COL].agg(lambda x: ' > '.join(x.astype(str)))
    activity_variant_counts = activity_variant_series.value_counts()
    activity_variant_table = activity_variant_counts.head(300).reset_index()
    activity_variant_table.columns = ['activity_variant', 'case_count']
    activity_variant_table['share_pct'] = activity_variant_table['case_count'] / len(case_df) * 100
    activity_variant_table['cum_share_pct'] = activity_variant_table['share_pct'].cumsum()
    save_csv(activity_variant_table, '24_top_300_activity_variants.csv', index=False)
    coverage = (activity_variant_counts.cumsum() / activity_variant_counts.sum()).reset_index(drop=True)
    coverage_df = pd.DataFrame({'variant_rank': np.arange(1, len(coverage) + 1), 'cumulative_case_share_pct': coverage.values * 100})
    save_csv(coverage_df.head(5000), '25_activity_variant_coverage_curve_first5000.csv', index=False)
    variant_case_df = pd.DataFrame({'activity_variant': activity_variant_series})
    variant_case_df[PRIMARY_LABEL] = variant_case_df.index.map(label_map)
    variant_group_summary = []
    for val, label_name in [(False, 'standard'), (True, 'scd')]:
        s = variant_case_df.loc[variant_case_df[PRIMARY_LABEL] == val, 'activity_variant']
        vc = s.value_counts()
        variant_group_summary.append({'group': label_name, 'n_cases': int(len(s)), 'n_unique_variants': int(vc.shape[0]), 'top1_variant_share_pct': round(float(vc.iloc[0] / len(s) * 100), 4) if len(vc) else np.nan, 'top10_variant_share_pct': round(float(vc.head(10).sum() / len(s) * 100), 4) if len(vc) else np.nan, 'variants_needed_for_50_pct': int((vc.cumsum() / vc.sum()).lt(0.5).sum() + 1) if len(vc) else np.nan, 'variants_needed_for_80_pct': int((vc.cumsum() / vc.sum()).lt(0.8).sum() + 1) if len(vc) else np.nan})
    variant_group_summary = pd.DataFrame(variant_group_summary)
    save_csv(variant_group_summary, '26_activity_variant_summary_by_primary_label.csv', index=False)
    display(activity_variant_table.head(20))
    display(variant_group_summary)
else:
    print('BUILD_ACTIVITY_VARIANTS=False; überspringe Variantenanalyse.')


In [ ]:
# Kombinierte Varianten
if BUILD_COMBINED_VARIANTS:
    print('Baue Combined-Activity-Varianten. Das kann deutlich länger dauern ...')
    combined_variant_series = seq_df.groupby(CASE_COL, sort=False)['combined_activity'].agg(lambda x: ' > '.join(x.astype(str)))
    combined_variant_counts = combined_variant_series.value_counts()
    combined_variant_table = combined_variant_counts.head(100).reset_index()
    combined_variant_table.columns = ['combined_variant', 'case_count']
    combined_variant_table['share_pct'] = combined_variant_table['case_count'] / len(case_df) * 100
    combined_variant_table['cum_share_pct'] = combined_variant_table['share_pct'].cumsum()
    save_csv(combined_variant_table, '27_top_100_combined_variants.csv', index=False)
    display(combined_variant_table.head(10))
else:
    print('Kombinierte Varianten sind deaktiviert.')


In [ ]:
# Direktfolgebeziehungen
if BUILD_DIRECT_FOLLOWS:
    print('Berechne technische Direct-Follows-Pairs ...')
    seq_small = df[[CASE_COL, TIME_COL, '__row_order__', ACTIVITY_COL, 'combined_activity']].sort_values([CASE_COL, TIME_COL, '__row_order__'], kind='mergesort').copy()
    seq_small['next_activity'] = seq_small.groupby(CASE_COL)[ACTIVITY_COL].shift(-1)
    seq_small['next_combined_activity'] = seq_small.groupby(CASE_COL)['combined_activity'].shift(-1)
    seq_small['next_timestamp'] = seq_small.groupby(CASE_COL)[TIME_COL].shift(-1)
    seq_small['delta_hours_to_next'] = (seq_small['next_timestamp'] - seq_small[TIME_COL]).dt.total_seconds() / 3600
    seq_small = seq_small.dropna(subset=['next_activity'])
    seq_small[PRIMARY_LABEL] = seq_small[CASE_COL].map(label_map).astype(bool)
    seq_small['same_timestamp_transition'] = seq_small['delta_hours_to_next'].eq(0)
    dfg_activity = seq_small.groupby([PRIMARY_LABEL, ACTIVITY_COL, 'next_activity']).size().reset_index(name='pair_count')
    dfg_activity['group_total_pairs'] = dfg_activity.groupby(PRIMARY_LABEL)['pair_count'].transform('sum')
    dfg_activity['pair_share_within_group_pct'] = dfg_activity['pair_count'] / dfg_activity['group_total_pairs'] * 100
    save_csv(dfg_activity.sort_values([PRIMARY_LABEL, 'pair_count'], ascending=[True, False]).head(1000), '28_top_direct_follows_activity_by_primary_label.csv', index=False)
    if BUILD_COMBINED_DIRECT_FOLLOWS:
        dfg_combined = seq_small.groupby([PRIMARY_LABEL, 'combined_activity', 'next_combined_activity']).size().reset_index(name='pair_count')
        dfg_combined['group_total_pairs'] = dfg_combined.groupby(PRIMARY_LABEL)['pair_count'].transform('sum')
        dfg_combined['pair_share_within_group_pct'] = dfg_combined['pair_count'] / dfg_combined['group_total_pairs'] * 100
        save_csv(dfg_combined.sort_values([PRIMARY_LABEL, 'pair_count'], ascending=[True, False]).head(1000), '29_top_direct_follows_combined_by_primary_label.csv', index=False)
    else:
        print('Kombinierte Direktfolgen sind deaktiviert.')
    tie_transition_summary = seq_small.groupby(PRIMARY_LABEL)['same_timestamp_transition'].agg(n_pairs='size', n_same_timestamp='sum', share_same_timestamp_pct=lambda x: round(float(x.mean() * 100), 4)).reset_index()
    save_csv(tie_transition_summary, '30_direct_follows_same_timestamp_summary.csv', index=False)
    display(tie_transition_summary)
else:
    print('BUILD_DIRECT_FOLLOWS=False; überspringe Direct-Follows.')


In [ ]:
# Teilprozesslaufzeiten
if 'subprocess' in df.columns:
    sp = df.groupby([CASE_COL, 'subprocess'])[TIME_COL].agg(first_ts='min', last_ts='max', event_count='size').reset_index()
    sp['span_days'] = (sp['last_ts'] - sp['first_ts']).dt.total_seconds() / (3600 * 24)
    sp[PRIMARY_LABEL] = sp[CASE_COL].map(label_map).astype(bool)
    subprocess_summary = sp.groupby([PRIMARY_LABEL, 'subprocess']).agg(n_cases=(CASE_COL, 'nunique'), total_events=('event_count', 'sum'), median_events_per_case_subprocess=('event_count', 'median'), median_span_days=('span_days', 'median'), p90_span_days=('span_days', lambda x: x.quantile(0.9))).reset_index()
    save_csv(subprocess_summary, '31_subprocess_span_summary_by_primary_label.csv', index=False)
    display(subprocess_summary.sort_values([PRIMARY_LABEL, 'n_cases'], ascending=[True, False]).head(30))
else:
    print('Keine subprocess-Spalte verfügbar.')


In [ ]:
# Fallattribute
case_attr_cols_existing = [c for c in case_df.columns if c.startswith('case:')]
numeric_attr_rows = []
for c in case_attr_cols_existing:
    converted = pd.to_numeric(case_df[c], errors='coerce')
    non_missing_rate = converted.notna().mean()
    if non_missing_rate >= 0.7 and converted.nunique(dropna=True) > 2:
        pos = converted[case_df[PRIMARY_LABEL]]
        neg = converted[~case_df[PRIMARY_LABEL]]
        numeric_attr_rows.append({'attribute': c, 'non_missing_rate': round(float(non_missing_rate), 4), 'nunique_numeric': int(converted.nunique(dropna=True)), 'median_scd': round(float(pos.median()), 4), 'median_standard': round(float(neg.median()), 4), 'median_diff': round(float(pos.median() - neg.median()), 4), 'mean_scd': round(float(pos.mean()), 4), 'mean_standard': round(float(neg.mean()), 4), 'smd_mean_scd_standard': round(float(standardised_mean_diff(pos, neg)), 4) if not pd.isna(standardised_mean_diff(pos, neg)) else np.nan})
numeric_attr_comparison = pd.DataFrame(numeric_attr_rows)
if len(numeric_attr_comparison):
    numeric_attr_comparison = numeric_attr_comparison.sort_values('smd_mean_scd_standard', key=lambda s: s.abs(), ascending=False)
    save_csv(numeric_attr_comparison, '32_numeric_case_attribute_comparison_primary_label.csv', index=False)
    display(numeric_attr_comparison.head(30))
else:
    print('Keine geeigneten numerischen Case-Attribute gefunden.')


In [ ]:
# Kategoriale Fallattribute
categorical_rows = []
for c in case_attr_cols_existing:
    nunique = case_df[c].nunique(dropna=False)
    if 2 <= nunique <= 20:
        tmp = pd.crosstab(case_df[c], case_df[PRIMARY_LABEL], normalize='columns') * 100
        if False in tmp.columns and True in tmp.columns:
            tmp['difference_pp_scd_minus_standard'] = tmp[True] - tmp[False]
            top_abs = tmp['difference_pp_scd_minus_standard'].abs().max()
            top_value = tmp['difference_pp_scd_minus_standard'].abs().idxmax()
            categorical_rows.append({'attribute': c, 'nunique': int(nunique), 'max_abs_difference_pp': round(float(top_abs), 4), 'value_with_max_difference': str(top_value)})
categorical_attr_summary = pd.DataFrame(categorical_rows).sort_values('max_abs_difference_pp', ascending=False) if categorical_rows else pd.DataFrame()
if len(categorical_attr_summary):
    save_csv(categorical_attr_summary, '33_categorical_case_attribute_difference_summary.csv', index=False)
    display(categorical_attr_summary.head(30))
else:
    print('Keine geeigneten kategorialen Case-Attribute gefunden.')


In [ ]:
# Abbildungen
fig, ax = plt.subplots(figsize=(9, 6))
plot_df = label_prevalence.sort_values('share_pct', ascending=True)
ax.barh(plot_df['label'], plot_df['share_pct'])
ax.set_xlabel('Anteil Fälle (%)')
ax.set_title('Prävalenz der Label-Kandidaten')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_01_label_prevalence.png')
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(case_df['duration_days'].dropna(), bins=60)
ax.axvline(thresholds['duration_p90'], linestyle='--', label='P90')
ax.axvline(thresholds['duration_p95'], linestyle=':', label='P95')
ax.set_xlabel('Durchlaufzeit pro Case (Tage)')
ax.set_ylabel('Anzahl Cases')
ax.set_title('Verteilung der Case Durations')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_02_case_duration_distribution.png')
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(case_df['event_count'].dropna(), bins=60)
ax.axvline(thresholds['event_count_p90'], linestyle='--', label='P90')
ax.axvline(thresholds['event_count_p95'], linestyle=':', label='P95')
ax.set_xlabel('Events pro Case')
ax.set_ylabel('Anzahl Cases')
ax.set_title('Verteilung der Case Lengths')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_03_case_length_distribution.png')
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(case_df['combined_rework_extra'].dropna(), bins=60)
ax.axvline(thresholds['combined_rework_p90'], linestyle='--', label='P90')
ax.axvline(thresholds['combined_rework_p95'], linestyle=':', label='P95')
ax.set_xlabel('Combined Rework Extra pro Case')
ax.set_ylabel('Anzahl Cases')
ax.set_title('Verteilung der kontextualisierten Rework-Intensität')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_04_combined_rework_distribution.png')
if len(label_by_case_year):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(label_by_case_year[CASE_YEAR_COL].astype(str), label_by_case_year['share_positive_pct'])
    ax.set_xlabel('case:year')
    ax.set_ylabel('SCD-P90-OR Anteil (%)')
    ax.set_title('SCD-P90-OR nach case:year')
    ax.grid(axis='y', alpha=0.3)
    save_fig(fig, 'fig_05_primary_label_by_case_year.png')

def boxplot_with_labels(ax, data, labels, showfliers=False):
    try:
        return ax.boxplot(data, tick_labels=labels, showfliers=showfliers)
    except TypeError:
        return ax.boxplot(data, labels=labels, showfliers=showfliers)
primary_mask = case_df[PRIMARY_LABEL].fillna(False).astype(bool)
for metric, filename, title, ylabel in [('duration_days', 'fig_06_duration_by_primary_label.png', 'Duration: Standard vs SCD', 'Durchlaufzeit (Tage)'), ('event_count', 'fig_07_event_count_by_primary_label.png', 'Event Count: Standard vs SCD', 'Events pro Case'), ('combined_rework_extra', 'fig_08_combined_rework_by_primary_label.png', 'Combined Rework: Standard vs SCD', 'Combined Rework Extra'), ('n_doctypes', 'fig_09_doctypes_by_primary_label.png', 'Document Types: Standard vs SCD', 'Anzahl Document Types'), ('n_subprocesses', 'fig_10_subprocesses_by_primary_label.png', 'Subprocesses: Standard vs SCD', 'Anzahl Subprocesses')]:
    if metric in case_df.columns:
        standard_values = pd.to_numeric(case_df.loc[~primary_mask, metric], errors='coerce').dropna()
        scd_values = pd.to_numeric(case_df.loc[primary_mask, metric], errors='coerce').dropna()
        if len(standard_values) == 0 or len(scd_values) == 0:
            print(f'Übersprungen: {metric}, weil eine Gruppe leer ist.')
            continue
        fig, ax = plt.subplots(figsize=(7, 5))
        data = [standard_values, scd_values]
        boxplot_with_labels(ax, data, labels=['Standard', 'SCD'], showfliers=False)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(axis='y', alpha=0.3)
        save_fig(fig, filename)
if BUILD_ACTIVITY_VARIANTS:
    fig, ax = plt.subplots(figsize=(8, 5))
    to_plot = coverage_df.head(5000)
    ax.plot(to_plot['variant_rank'], to_plot['cumulative_case_share_pct'])
    ax.set_xlabel('Variantenrang')
    ax.set_ylabel('Kumulative Fallabdeckung (%)')
    ax.set_title('Variant Coverage Curve')
    ax.grid(alpha=0.3)
    save_fig(fig, 'fig_11_variant_coverage_curve.png')
if 'wide_presence' in globals() and len(wide_presence) > 0:
    fig, ax = plt.subplots(figsize=(9, 7))
    top_diff = wide_presence.head(20).sort_values('difference_pp_scd_minus_standard')
    ax.barh(top_diff['combined_activity'], top_diff['difference_pp_scd_minus_standard'])
    ax.set_xlabel('Differenz Case Share SCD - Standard (Prozentpunkte)')
    ax.set_title('Combined Activities, die in SCD-Fällen häufiger vorkommen')
    ax.grid(axis='x', alpha=0.3)
    save_fig(fig, 'fig_12_top_combined_activity_difference_scd.png')
else:
    print('Figur 14.8 übersprungen: wide_presence ist nicht vorhanden oder leer.')
if 'doctype' in event_df.columns and 'subprocess' in event_df.columns:
    matrix = pd.crosstab(event_df['doctype'], event_df['subprocess'])
    if matrix.shape[0] > 0 and matrix.shape[1] > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        im = ax.imshow(matrix.values, aspect='auto')
        ax.set_xticks(np.arange(matrix.shape[1]))
        ax.set_yticks(np.arange(matrix.shape[0]))
        ax.set_xticklabels(matrix.columns, rotation=45, ha='right')
        ax.set_yticklabels(matrix.index)
        ax.set_title('Event-Verteilung: doctype × subprocess')
        fig.colorbar(im, ax=ax, label='Event Count')
        save_fig(fig, 'fig_13_doctype_subprocess_heatmap.png')
    else:
        print('Figur 14.9 übersprungen: doctype × subprocess Matrix ist leer.')
else:
    print('Figur 14.9 übersprungen: doctype oder subprocess fehlt.')
required_cols_scatter = ['event_count', 'combined_rework_extra']
if all((col in case_df.columns for col in required_cols_scatter)):
    scatter_df = case_df[required_cols_scatter].copy()
    scatter_df['event_count'] = pd.to_numeric(scatter_df['event_count'], errors='coerce')
    scatter_df['combined_rework_extra'] = pd.to_numeric(scatter_df['combined_rework_extra'], errors='coerce')
    scatter_df = scatter_df.dropna()
    if len(scatter_df) > 0:
        sample_n = min(12000, len(scatter_df))
        sample_df = scatter_df.sample(sample_n, random_state=42) if len(scatter_df) > sample_n else scatter_df
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(sample_df['event_count'], sample_df['combined_rework_extra'], alpha=0.25, s=8)
        ax.axvline(thresholds['event_count_p90'], linestyle='--')
        ax.axhline(thresholds['combined_rework_p90'], linestyle='--')
        ax.set_xlabel('Events pro Case')
        ax.set_ylabel('Combined Rework Extra')
        ax.set_title('SCD-Komponenten: Event Count vs. Combined Rework')
        ax.grid(alpha=0.3)
        save_fig(fig, 'fig_14_scd_components_scatter.png')
    else:
        print('Figur 14.10 übersprungen: keine gültigen Scatter-Werte vorhanden.')
else:
    print('Figur 14.10 übersprungen: event_count oder combined_rework_extra fehlt.')
print('Abbildungen erstellt:', len(created_figures))


In [ ]:
# Falldaten speichern
case_df_export_cols = ['case_start', 'case_end', 'duration_days', 'event_count', 'raw_rework_extra', 'combined_rework_extra', 'combined_rework_share_of_events', 'n_activity_labels', 'n_combined_activity_labels', 'n_doctypes', 'n_subprocesses', 'n_docids', 'n_resources', PRIMARY_LABEL, ROBUSTNESS_LABEL, 'label_temporal_duration_p90', 'label_path_change_or_objection', 'label_remove_document']
case_df_export_cols = [c for c in case_df_export_cols if c in case_df.columns]
for c in [CASE_YEAR_COL, CASE_DEPARTMENT_COL, CASE_APPLICANT_COL, CASE_REJECTED_COL]:
    if c is not None and c in case_df.columns and (c not in case_df_export_cols):
        case_df_export_cols.append(c)
case_df[case_df_export_cols].to_csv(TABLE_DIR / '35_case_level_features_descriptive_analysis.csv', encoding='utf-8-sig')
created_tables.append(TABLE_DIR / '35_case_level_features_descriptive_analysis.csv')
